## 1. Loading the dataset into a pandas dataframe and seeing its dimensions


In [ ]:
import pandas as pd

df_ais = pd.read_csv('../data/raw/aisdk-2024-08-07.csv')
print(df_ais.shape)

The dataset contains 28,203,137 AIS records and 26 columns representing 24 hours of vessel activity 

## 2. Inspecting the dataset column structure

In [ ]:
print(df_ais.columns)

The dataset contains 26 columns. The ones most relevant to this project are: 
- 'MMSI' -> unique vessel identifier
- 'Latitude' and 'Longitude' -> vessel position
- 'SOG' -> speed over ground in knots(kn)
- 'Ship type' -> vessel category
- 'Destination' -> reported destination(where available)
- 'ETA' -> reported ETA(where available)
- '# Timestamp' -> timestamp from the AIS basestation
- 'Type of mobile' -> type of target this message is received from (class A AIS Vessel, Class B AIS vessel, etc)
- 'Navigational status' -> navigational status from AIS message (where available), e.g.: 'Engaged in fishing', 'Under way using engine'
- 'Name' -> vessel name 

## 3. Exploring vessel records and target vessel types
Since the dataset has multiple million observations we will find:
- unique vessel MMSI's
- observations per unique MMSI
- unique vessel types(we target cargo, tanker and passenger)
- destination and ETA availability across unique MMSI's with our target type

In [ ]:
df_vessels = df_ais['MMSI'].nunique()
print(df_vessels)

df_obs = df_ais.groupby('MMSI').size()
print(df_obs)

vessel_obs = df_ais[df_ais['Type of mobile'] == 'Class A'].groupby('MMSI').size()
print(vessel_obs)

There are 9102 unique MMSI's in the dataset.  
Some MMSI's, such as 148 and 3638, appear unusually short or non-standard. Since the analysis focuses on commercial cargo, tanker, and passenger vessels, Type of mobile = Class A is relevant, but unusual MMSI values still remain even within Class A records.

In [ ]:
df_types = df_ais.groupby('Ship type').size()
print(df_types)

The raw dataset contains:
- 5,628,117 cargo observations
- 2,647,622 tanker observations
- 3,566,724 passenger observations

In [ ]:
unique_mmsi_by_type = df_ais.groupby('Ship type')['MMSI'].nunique()
print(unique_mmsi_by_type)

Among the target vessel types the dataset contains:
- 694 cargo MMSI's
- 260 tanker MMSI's
- 435 passenger MMSI's  
- Keep in mind each MMSI does not necessarily represent an actual vessel

In [ ]:
df_target = df_ais[df_ais['Ship type'].isin(['Cargo', 'Tanker', 'Passenger'])]
eta_availability = df_target.groupby('Ship type')['ETA'].count()
dest_availability = df_target.groupby('Ship type')['Destination'].count()
print(eta_availability)
print(dest_availability)

ETA availability:
- 5,522,543 cargo observations
- 2,793,477 passenger observations
- 2,627,667 tanker observations

Destination availability:
- 5,628,117 cargo observations
- 3,566,724 passenger observations
- 2,647,622 tanker observations  
- We've established that all target observations have a non-null destination field

Sidenote: non-null observations such as 'Unknown' do not necessarily count as valid ones

## 4. Absence and field quality of key values 
We will investigate:  
- MMSI  
- Latitude  
- Longitude
- SOG  
- Navigational Status
- Most common destinations  
- ETA format and validity  
- Navigational Status validity  
- SOG values and suspicious speeds

In [ ]:
mmsi_count = df_ais['MMSI'].notnull().sum()
print(mmsi_count)

lat_count = df_ais['Latitude'].notnull().sum()
lon_count = df_ais['Longitude'].notnull().sum()
print(lat_count)
print(lon_count)

sog_count = df_ais['SOG'].notnull().sum()
print(sog_count)

nav_status_count = df_ais['Navigational status'].notnull().sum()
print(nav_status_count)

MMSI, Latitude, Longitude, and Navigational Status are complete across the dataset. SOG contains missing values(around 2.2 million rows) and will require handling during the cleaning stage

In [ ]:
df_dest = df_target['Destination'].value_counts(ascending=False).head(30)
print(df_dest)

Destination values and their format vary. Many use UN/LOCODE-style codes, while others use free-text port names, route strings, or non-specific values such as Unknown and FOR ORDERS. Destination standardization will be required before port-coordinate mapping. There are 362,278 observations with the value 'Unknown'. These observations do not necessarily represent a major issue for the analysis since they could come from a small numbers of really 'chatty' vessels with 'Unknown' values. We will need to check how many unique MMSI's each destination has

In [ ]:
df_dest_unique_mmsi = df_target.groupby('Destination')['MMSI'].nunique().sort_values(ascending=False).head(30)
print(df_dest_unique_mmsi)

After further investigation 'Unknown' dominates with 153 unique MMSI's. Furthermore destination values are highly inconsistent and each destination appears under multiple names e.g. SE GOT / SEGOT,
DK SKA / DKSKA / SKAGEN,
FOR ORDERS / FOR ORDER,
PLGDN / PL GDN,
DERSK / DE RSK

In [ ]:
df_eta = df_target['ETA'].dropna().head(30)
print(df_eta)

The ETA field contains full datetime values and is often repeated across multiple AIS observations for the same vessel.
Because this project measures vessel progress during the 24-hour tracking period, the ETA does not need to fall within or close to the observed 24-hour window. Vessels may be at very different stages of their voyages

In [ ]:
df_nav_status = df_target['Navigational status'].value_counts()
print(df_nav_status)

Most target vessel observations are classified as 'Under way using engine' with a precise count of 10.8 million observations.
Other relevant states include:
- 'Constrained by her draught'-  280,034 observations
- 'At anchor'- 213,377 observations
- 'Moored'-  203,910 observations
- 'Restricted maneuverability'-  47,954 observations

In [ ]:
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
print(df_target['SOG'].describe())

Speed values are as expected for commercial vessels with the maximum of 99.7 knots being invalid/suspicious